# Task 1: Data Cleaning & Preprocessing
**Dataset:** Netflix titles (`Dataset.csv`)  
**Goal:** Turn the raw data into a clean, analysis-ready dataset.

**Workflow:** 1) Import → 2) Identify missing & duplicate records → 3) Handle nulls & inconsistent formats → 4) Transform categorical & date columns → 5) Save clean dataset

## Step 1: Import the dataset

In [1]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv('Dataset.csv')
df = df_raw.copy()   # work on a copy, keep the raw data untouched

print('Shape:', df.shape)
df.head()

Shape: (8790, 10)


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8790 non-null   str  
 1   type          8790 non-null   str  
 2   title         8790 non-null   str  
 3   director      8790 non-null   str  
 4   country       8790 non-null   str  
 5   date_added    8790 non-null   str  
 6   release_year  8790 non-null   int64
 7   rating        8790 non-null   str  
 8   duration      8790 non-null   str  
 9   listed_in     8790 non-null   str  
dtypes: int64(1), str(9)
memory usage: 686.8 KB


## Step 2: Identify missing and duplicate records
`isna()` shows no nulls, but that can be misleading: missing values are often hidden as placeholder text. Let's check for those too.

In [3]:
print('Real nulls per column:')
print(df.isna().sum())

placeholders = ['Not Given', 'Unknown', 'N/A', 'NA', 'None', '']
hidden = {c: int(df[c].isin(placeholders).sum()) for c in df.columns}
hidden = {c: n for c, n in hidden.items() if n}
print('\nHidden missing values (placeholder text):', hidden)
print('\nShare of rows affected:')
print((pd.Series(hidden) / len(df) * 100).round(1).astype(str) + ' %')

Real nulls per column:
show_id         0
type            0
title           0
director        0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
dtype: int64

Hidden missing values (placeholder text): {'title': 1, 'director': 2588, 'country': 287}

Share of rows affected:
title        0.0 %
director    29.4 %
country      3.3 %
dtype: str


In [4]:
# Duplicates: exact rows, duplicate IDs, and duplicates ignoring the ID column
print('Exact duplicate rows      :', df.duplicated().sum())
print('Duplicate show_id         :', df['show_id'].duplicated().sum())

cols_wo_id = [c for c in df.columns if c != 'show_id']
print('Duplicates ignoring show_id:', df.duplicated(subset=cols_wo_id).sum())

df[df.duplicated(subset=cols_wo_id, keep=False)].sort_values('title')

Exact duplicate rows      : 0
Duplicate show_id         : 0
Duplicates ignoring show_id: 3


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
2925,s3963,Movie,15-Aug,Swapnaneel Jayakar,India,3/29/2019,2019,TV-14,124 min,"Comedies, Dramas, Independent Movies"
4261,s5967,Movie,15-Aug,Swapnaneel Jayakar,India,3/29/2019,2019,TV-14,124 min,"Comedies, Dramas, Independent Movies"
3285,s4523,Movie,22-Jul,Paul Greengrass,Norway,10/10/2018,2018,R,144 min,"Dramas, Thrillers"
4260,s5966,Movie,22-Jul,Paul Greengrass,Norway,10/10/2018,2018,R,144 min,"Dramas, Thrillers"
393,s3997,TV Show,9-Feb,Not Given,Pakistan,3/20/2019,2018,TV-14,1 Season,"International TV Shows, TV Dramas"
537,s5965,TV Show,9-Feb,Not Given,Pakistan,3/20/2019,2018,TV-14,1 Season,"International TV Shows, TV Dramas"


**Finding:** 3 records are repeated under a different `show_id` (same title, country, date added, etc.), so they are true duplicates. `director` and `country` carry hidden missing values as `'Not Given'`.

## Step 3: Handle nulls, duplicates and inconsistent formats

In [5]:
# 3a. Remove duplicates (keep first occurrence)
before = len(df)
df = df.drop_duplicates(subset=cols_wo_id, keep='first').reset_index(drop=True)
print(f'Removed {before - len(df)} duplicate rows -> {len(df)} remain')

Removed 3 duplicate rows -> 8787 remain


In [6]:
# 3b. Convert placeholder text to real NaN, then fill with an explicit label.
# We do NOT drop these rows: 29% of titles have no director, and dropping would bias the data.
for col in ['director', 'country']:
    df[col] = df[col].replace(placeholders, np.nan)

print(df[['director', 'country']].isna().sum())

df['director'] = df['director'].fillna('Unknown')
df['country'] = df['country'].fillna('Unknown')
print('\nRemaining nulls:', df.isna().sum().sum())

director    2587
country      287
dtype: int64

Remaining nulls: 0


In [7]:
# 3c. Consistency checks: whitespace, spelling variants of categories
for col in df.select_dtypes(include='object').columns.tolist() + df.select_dtypes(include='string').columns.tolist():
    df[col] = df[col].str.strip()

print(df['type'].value_counts(), '\n')
print(df['rating'].value_counts())

type
Movie      6124
TV Show    2663
Name: count, dtype: int64 

rating
TV-MA       3205
TV-14       2155
TV-PG        861
R            798
PG-13        490
TV-Y7        333
TV-Y         306
PG           287
TV-G         220
NR            79
G             41
TV-Y7-FV       6
NC-17          3
UR             3
Name: count, dtype: int64


/tmp/ipykernel_173/1341515226.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns.tolist() + df.select_dtypes(include='string').columns.tolist():


In [8]:
# 3d. Corrupted titles: Excel auto-converted some titles into dates (e.g. '22 July' -> '22-Jul')
import re
suspect = df[df['title'].str.fullmatch(r'\d{1,2}-(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)')]
suspect[['show_id', 'type', 'title', 'country', 'release_year']]

,show_id,type,title,country,release_year
393,s3997,TV Show,9-Feb,Pakistan,2018
2924,s3963,Movie,15-Aug,India,2019
3284,s4523,Movie,22-Jul,Norway,2018
4258,s5964,Movie,1-Oct,Nigeria,2014


These few titles look like they were date-converted by Excel before the file reached us. The original wording can't be recovered with certainty from the data alone, so I flag them and leave them unchanged.

## Step 4: Transform categorical and date-related columns

In [9]:
# 4a. Dates: text 'm/d/yyyy' -> real datetime, plus useful parts
df['date_added'] = pd.to_datetime(df['date_added'], format='%m/%d/%Y')

df['year_added']  = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month
df['month_name']  = df['date_added'].dt.month_name()
df['weekday_added'] = df['date_added'].dt.day_name()

# How long after release did the title reach Netflix?
df['years_to_netflix'] = df['year_added'] - df['release_year']
print(df['years_to_netflix'].describe())
print('Titles added BEFORE their release year:', (df['years_to_netflix'] < 0).sum())

count    8787.000000
mean        4.691931
std         8.791157
min        -3.000000
25%         0.000000
50%         1.000000
75%         5.000000
max        93.000000
Name: years_to_netflix, dtype: float64
Titles added BEFORE their release year: 14


In [10]:
# 4b. Duration: '90 min' / '3 Seasons' -> number + unit, split by type
dur = df['duration'].str.extract(r'(?P<value>\d+)\s*(?P<unit>\D+)')
df['duration_value'] = dur['value'].astype(int)

df['duration_min'] = np.where(df['type'] == 'Movie', df['duration_value'], np.nan)
df['seasons']      = np.where(df['type'] == 'TV Show', df['duration_value'], np.nan)
df = df.drop(columns=['duration_value'])

print(df.groupby('type')[['duration_min', 'seasons']].describe().T)

type                      Movie      TV Show
duration_min count  6124.000000     0.000000
             mean     99.573645          NaN
             std      28.280425          NaN
             min       3.000000          NaN
             25%      87.000000          NaN
             50%      98.000000          NaN
             75%     114.000000          NaN
             max     312.000000          NaN
seasons      count     0.000000  2663.000000
             mean           NaN     1.752159
             std            NaN     1.550844
             min            NaN     1.000000
             25%            NaN     1.000000
             50%            NaN     1.000000
             75%            NaN     2.000000
             max            NaN    17.000000


In [11]:
# 4c. Genres: 'listed_in' holds several genres in one cell
df['genre_count']   = df['listed_in'].str.split(', ').str.len()
df['primary_genre'] = df['listed_in'].str.split(', ').str[0]

df[['title', 'listed_in', 'primary_genre', 'genre_count']].head()

,title,listed_in,primary_genre,genre_count
0,Dick Johnson Is Dead,Documentaries,Documentaries,1
1,Ganglands,"Crime TV Shows, International TV Shows, TV Act...",Crime TV Shows,3
2,Midnight Mass,"TV Dramas, TV Horror, TV Mysteries",TV Dramas,3
3,Confessions of an Invisible Girl,"Children & Family Movies, Comedies",Children & Family Movies,2
4,Sankofa,"Dramas, Independent Movies, International Movies",Dramas,3


In [12]:
# 4d. Ratings: group the 14 labels into 4 audience levels
rating_map = {
    'TV-Y': 'Kids', 'TV-Y7': 'Kids', 'TV-Y7-FV': 'Kids', 'TV-G': 'Kids', 'G': 'Kids',
    'TV-PG': 'Older Kids', 'PG': 'Older Kids',
    'TV-14': 'Teens', 'PG-13': 'Teens',
    'TV-MA': 'Adults', 'R': 'Adults', 'NC-17': 'Adults',
    'NR': 'Not Rated', 'UR': 'Not Rated',
}
df['rating_group'] = df['rating'].map(rating_map)
print('Unmapped ratings:', df['rating_group'].isna().sum())
df['rating_group'].value_counts()

Unmapped ratings: 0


rating_group
Adults        4006
Teens         2645
Older Kids    1148
Kids           906
Not Rated       82
Name: count, dtype: int64

In [13]:
# 4e. Memory-efficient categorical dtypes
for col in ['type', 'rating', 'rating_group', 'month_name', 'weekday_added']:
    df[col] = df[col].astype('category')

df.dtypes

show_id                        str
type                      category
title                          str
director                       str
country                        str
date_added          datetime64[us]
release_year                 int64
rating                    category
duration                       str
listed_in                      str
year_added                   int32
month_added                  int32
month_name                category
weekday_added             category
years_to_netflix             int64
duration_min               float64
seasons                    float64
genre_count                  int64
primary_genre               object
rating_group              category
dtype: object

## Step 5: Create the clean dataset

In [14]:
# Final validation
assert df['show_id'].is_unique
assert df[['type', 'title', 'date_added', 'release_year', 'rating']].isna().sum().sum() == 0
assert df['date_added'].dtype == 'datetime64[ns]' or str(df['date_added'].dtype).startswith('datetime64')

print('Raw  shape:', df_raw.shape)
print('Clean shape:', df.shape)
df.head()

Raw  shape: (8790, 10)
Clean shape: (8787, 20)


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in,year_added,month_added,month_name,weekday_added,years_to_netflix,duration_min,seasons,genre_count,primary_genre,rating_group
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,2021-09-25,2020,PG-13,90 min,Documentaries,2021,9,September,Saturday,1,90.0,NaN,1,Documentaries,Teens
1,s3,TV Show,Ganglands,Julien Leclercq,France,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",2021,9,September,Friday,0,NaN,1.0,3,Crime TV Shows,Adults
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,2021-09-24,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries",2021,9,September,Friday,0,NaN,1.0,3,TV Dramas,Adults
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,2021-09-22,2021,TV-PG,91 min,"Children & Family Movies, Comedies",2021,9,September,Wednesday,0,91.0,NaN,2,Children & Family Movies,Older Kids
4,s8,Movie,Sankofa,Haile Gerima,United States,2021-09-24,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies",2021,9,September,Friday,28,125.0,NaN,3,Dramas,Adults


In [15]:
df.to_csv('netflix_clean.csv', index=False)
print('Saved netflix_clean.csv')

Saved netflix_clean.csv


## Summary of cleaning steps
| Issue found | Action |
|---|---|
| 3 duplicate records (different `show_id`) | Removed |
| `'Not Given'` placeholders in `director` and `country` | Converted to NaN, filled with `'Unknown'` (rows kept) |
| `date_added` stored as text | Converted to datetime; added year, month, month name, weekday |
| `duration` mixed minutes and seasons in one text column | Split into `duration_min` (movies) and `seasons` (TV shows) |
| Multiple genres in one cell | Added `primary_genre` and `genre_count` |
| 14 rating labels | Added `rating_group` (Kids / Older Kids / Teens / Adults / Not Rated) |
| Titles damaged by Excel date conversion | Flagged, left unchanged |
| Text categories | Stripped whitespace, converted to `category` dtype |